# Preprocessing BYOND by BSI

**Tahap:** 1-8 (Load → Filter Rating → Relative Time → Dedup → Text & Slang Normalization → Short Filter → Save)

**Input:** `data/raw/BYOND_by_BSI_raw.csv` (48,757 reviews)
**Output:** `data/processed/byond_bertopic.csv` & `byond_full.csv`

**Metodologi:**
- Filter rating 1-2 sebagai proxy keluhan teknis
- Relative month (1-12) dari launch date 9 November 2024
- Slang normalization: Salsabila lexicon + banking extension
- Stage 7 (language filter) di-deprecated berdasarkan audit pada wondr

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from datetime import datetime

from utils.preprocessing import (
    load_raw_reviews,
    filter_negative_ratings,
    add_relative_time_columns,
    drop_exact_duplicates,
    apply_normalization,
    load_slang_dict,
    apply_slang_normalization,
    filter_short_reviews,
    save_preprocessed_outputs,
)

# Configure pandas display
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 150)

## Tahap 1: Load Raw Reviews & Filter Rating 1-2

Filter rating 1-2 sebagai proxy keluhan teknis. Rating 3-5 di-drop dari analisis.

In [2]:
df = load_raw_reviews("data/raw/BYOND_by_BSI_raw.csv")
df = filter_negative_ratings(df)

✅ Loaded 48,757 reviews from data/raw/BYOND_by_BSI_raw.csv
✅ Rating filter: kept 22,935 / 48,757 reviews (dropped 25,822 non-negative reviews).
   Distribution: {1: 19290, 2: 3645}


## Tahap 2: Ekstraksi Relative Month & Week

- Launch date BYOND: 9 November 2024
- Window: 12 bulan (9 Nov 2024 - 8 Nov 2025)
- Relative month dihitung calendar-based (9 Nov - 8 Des = bulan 1, dst)

In [3]:
BYOND_LAUNCH = datetime(2024, 11, 9)
df = add_relative_time_columns(df, BYOND_LAUNCH)

✅ Relative time extraction (launch=2024-11-09, window=12 months):
   Kept 22,935 / 22,935 reviews in window.
   Distribution per relative_month: {1: 529, 2: 2240, 3: 2227, 4: 7539, 5: 1076, 6: 642, 7: 1353, 8: 1047, 9: 1382, 10: 3144, 11: 969, 12: 787}


## Tahap 3: Hapus Exact Duplicates

Dedup berdasarkan `review_text` (raw, sebelum normalisasi). Keep first
(earliest timestamp). Top 5 duplicates di-log untuk sanity check.

In [4]:
df = drop_exact_duplicates(df)

✅ Exact duplicate removal (by 'review_text', keep first):
   Kept 22,249 / 22,935 reviews (dropped 686 duplicates).

   Top 5 most-duplicated review texts:
     1. [37x] 'sering error'
     2. [33x] 'sering gangguan'
     3. [22x] 'Sering error'
     4. [21x] 'Sering eror'
     5. [20x] 'sering eror'


## Tahap 4: Text Normalization

Pipeline 8 step: lowercase → URL → emoji → number → repeat collapse → 
punctuation → single chars → whitespace.

**Catatan:** Stopword removal TIDAK dilakukan di sini (best practice
BERTopic — stopwords di-handle di tahap c-TF-IDF).

In [5]:
df = apply_normalization(df)

✅ Text normalization applied to 22,249 reviews.
   Empty after cleaning: 14 reviews (will be dropped at Stage 7 short-review filter).
   Word count (after) — mean: 17.7, median: 13, max: 96


## Tahap 5: Slang Normalization

Sumber kamus:
1. **Salsabila lexicon** (Salsabila et al. 2018) — base lexicon Bahasa Indonesia colloquial
2. **Banking extension** — 39 entries domain-specific (compiled manual)

**Filter applied:**
- 348 degrading reduplication mappings di-drop (mis. 'baru' → 'baru baru')
- 71 entri Salsabila di-blocklist untuk 6 kata baku Indonesia
  (`apa`, `nya`, `sekali`, `bener`, `minta`, `bukan`) yang ter-corrupt
  setelah preprocessing — diidentifikasi via data-driven inspection
  Top 30 most-triggered mappings.

In [6]:
slang_dict = load_slang_dict(
    salsabila_path="dictionaries/salsabila.csv",
    banking_ext_path="dictionaries/banking_extension.csv",
    # blocklist defaults to INDONESIAN_PROTECTED_WORDS in module
)

df = apply_slang_normalization(df, slang_dict)

   Dropped 348 degrading reduplication mappings (e.g., 'baru' → 'baru baru').
   Dropped 71 blocklisted entries (Indonesian standard words protected from mis-replacement).
✅ Loaded Salsabila lexicon: 3,641 entries (after preprocessing & dedup).
✅ Loaded banking extension: 51 entries.
✅ Merged dictionary: 3,682 total entries (10 banking entries override Salsabila).
✅ Slang normalization applied to 22,249 reviews.
   Reviews with at least one slang replacement: 16,970 (76.3%).
   Word count (after) — mean: 17.8, median: 13, max: 98


## Inspeksi Hasil


In [7]:
# Sample 10 review acak — sebelum vs sesudah preprocessing
df[["review_text", "review_text_cleaned", "word_count_after",
    "relative_month", "rating"]].sample(10, random_state=42)

,review_text,review_text_cleaned,word_count_after,relative_month,rating
10244,"Gausah banyak banyak gaya deh.... Mending bikin aplikasi simple, usefull, dan stabil kaya BCA .....",enggak usah banyak banyak gaya deh mending bikin aplikasi simple usefull dan stabil kayak bca,15,4,1
15713,"hilangin scan mukanya benar benar bikin kesal saja ini aplikasi ,cuman karna saya download ulang...",menghilangkan scan mukanya benar benar bikin kesal saja ini aplikasi cuman karena saya download ...,26,8,1
3532,"Aplikasi mempersulit pengguna BSI Mobile, aktifasi gak bisa² lambat. Mau transaksi BSI jadi gak ...",aplikasi mempersulit pengguna bsi mobile aktifasi tidak bisa lambat mau transaksi bsi jadi tidak...,20,3,1
11038,Gangguan mulu. App gak jelas. Mending app yg lama aja,gangguan terus aplikasi tidak jelas mending aplikasi yang lama saja,10,4,1
17211,"BSI gak jelas hari ini gak bisa tarik tunai dimana pun, enakan pas mandiri syariah aja,udah mute...",bsi tidak jelas hari ini tidak bisa tarik tunai dimana mana pun enakan pas mandiri syariah saja ...,24,9,1
2404,niatnya pingin ngikutin livin. tapi gak jelas. bukti transaksi gak bisa di download/dibagikan. m...,niatnya pengin mengikuti livin tapi tidak jelas bukti transaksi tidak bisa di download dibagikan...,27,2,1
4420,Sering gak bisa buka sampe sama instal ulang,sering tidak bisa buka sampai sama instal ulang,8,3,1
3211,Tolong ditambahkan jam dan pengirim pada menu mutasinya. Ga enak klo cuma bisa lihat tanggalnya ...,tolong ditambahkan jam dan pengirim pada menu mutasinya tidak enak kalo cuma bisa lihat tanggaln...,32,3,1
3950,Ga guna! Loading lama! Ga keliatan saldo rekening.! Parah! App rusak!,tidak guna loading lama tidak kelihatan saldo rekening parah aplikasi rusak,11,3,1
2564,Kenpa gak bisa isi saldo ke DANA gak kayak BSI mobile yg dulu.. Aneh katanya lebih canggih,kenpa tidak bisa isi saldo ke dana tidak kayak bsi mobile yang dulu aneh katanya lebih canggih,17,2,1


In [8]:
# Distribusi review per relative_month
print("Distribusi review per relative_month:")
print(df["relative_month"].value_counts().sort_index())
print(f"\nTotal reviews after Tahap 1-5: {len(df):,}")

Distribusi review per relative_month:
relative_month
1      529
2     2226
3     2199
4     7291
5     1053
6      629
7     1309
8     1008
9     1320
10    2993
11     932
12     760
Name: count, dtype: int64

Total reviews after Tahap 1-5: 22,249


## Tahap 6: Filter Short Reviews (<5 kata)

Drop reviews dengan word count <5 setelah Stage 4-5 normalization.
Diletakkan setelah slang normalization karena slang dapat expand short
forms (`mbanking` → `mobile banking`), mengubah word count.

Diletakkan **sebelum** Stage 7 (language filter) berdasarkan audit:
fasttext lid.176 punya akurasi rendah pada text <5 kata.

In [9]:
df = filter_short_reviews(df, min_words=5)

✅ Short review filter (min_words=5):
   Kept 19,393 / 22,249 reviews (dropped 2,856 reviews with < 5 words).
   Word count (after) — mean: 20.0, median: 15, max: 98


## Tahap 7: Language Filtering — DEPRECATED

Language filter (fasttext lid.176) **tidak digunakan** di pipeline final
berdasarkan data-driven audit:

- Iterasi 1: Tier 3 = 471 (4.6%), 5/5 sampel = false positive (Indonesian)
- Iterasi 2 (setelah reorder Stage 6 ↔ 7): Tier 3 = 123 (1.4%), tetap 9/10 false positive
- Pattern misclassification: banking loanwords → English, Indonesian-Malay similarity, typo

**Conclusion:** fasttext lid.176 punya systemic bias di domain banking
Indonesia. Asumsi domain (>95% Indonesian) dipertahankan; outlier
non-Indonesian akan ke-cluster sebagai outlier topic di BERTopic.

Detail audit: lihat `99_dev_wondr.ipynb` dan `preprocessing_methodology.md`.

## Tahap 8: Save Outputs (BERTopic-ready & Full Audit)

Save dua versi:
- **`wondr_bertopic.csv`** (slim) — kolom essential untuk pipeline BERTopic
- **`wondr_full.csv`** (audit) — semua kolom + metadata preprocessing

In [10]:
save_preprocessed_outputs(
       df,
       bertopic_path="data/processed/byond_bertopic.csv",   # <-- FIX
       full_path="data/processed/byond_full.csv",            # <-- FIX
   )

✅ BERTopic-ready saved: data/processed/byond_bertopic.csv
   Shape: (19393, 6), Columns: ['review_id', 'review_text_cleaned', 'relative_month', 'relative_week', 'date_wib', 'rating']

✅ Full audit saved: data/processed/byond_full.csv
   Shape: (19393, 20), Columns: 20 columns


## Validasi Akhir

In [11]:
# Validasi 1: Distribusi review per relative_month
print("Distribusi review per relative_month:")
print(df["relative_month"].value_counts().sort_index().to_string())
print(f"\nTotal final reviews: {len(df):,}")
print(f"Mean per month: {len(df) / 12:.0f}")

Distribusi review per relative_month:
relative_month
1      487
2     2040
3     1962
4     6284
5      921
6      543
7     1108
8      881
9     1125
10    2552
11     807
12     683

Total final reviews: 19,393
Mean per month: 1616


In [12]:
# Validasi 2: Sample 30 review (manual visual check)
sample = df.sample(30, random_state=42)
for _, row in sample.iterrows():
    print(f"[M{row['relative_month']:02d} R{row['rating']}] {row['review_text_cleaned'][:100]}")

[M03 R1] seperti belum siap launching sedikit error bermasalah terus yang tak kunjung terselesaikan membuat k
[M09 R1] tanggal agustus byound gangguan kah tidak bisa transaksi transaksi gagal tapi saldo berkurang
[M01 R2] bukti transfer tidak terbaca tidak disemua format terbaca jadi malah menyusahkan untuk kirim bukti t
[M04 R1] terlalu banyak update tidak jelas
[M11 R1] aplikasi rusak ketika kita sekarat dan butuh banget malah enggak bisa dipakai parah kalian bsi njenk
[M03 R1] menyusahkan orang kartu atm gua hilang buku tabungan di kampung bsi mobile yang lama tidak bisa di g
[M08 R1] kacau scan qris gagal terus loading lama tiba tiba keluar dan minta foto ulang qris ya lemot banget
[M10 R1] sudah berkali kali pakai tetap saja mau buka doang susah banget kenapa sih deh dibuka pakai paket in
[M08 R1] sudah hari ini tidak bisa buka beyond nya kendala terus
[M07 R1] mau dftr saja sush tolol kali
[M10 R1] lagi perlu nya dan genting nya malah lelet parah
[M03 R1] masih bagus aplikasi bsi

In [13]:
# Validasi 3: Top 100 most frequent words (untuk iterative refine banking dict)
from collections import Counter

all_words = " ".join(df["review_text_cleaned"].fillna("")).split()
word_counts = Counter(all_words)

print(f"Total unique words: {len(word_counts):,}")
print(f"Total tokens: {sum(word_counts.values()):,}")
print(f"\nTop 100 most frequent words:")
for i, (word, count) in enumerate(word_counts.most_common(100), 1):
    print(f"  {i:3d}. {word:25s} ({count:,})")

Total unique words: 12,358
Total tokens: 387,284

Top 100 most frequent words:
    1. aplikasi                  (13,635)
    2. tidak                     (13,447)
    3. bisa                      (9,151)
    4. di                        (7,212)
    5. ini                       (6,697)
    6. bsi                       (5,845)
    7. yang                      (5,376)
    8. sudah                     (5,055)
    9. mau                       (4,900)
   10. terus                     (4,684)
   11. nya                       (4,678)
   12. saya                      (4,295)
   13. saja                      (4,128)
   14. sering                    (3,831)
   15. byond                     (3,530)
   16. dan                       (3,453)
   17. error                     (3,441)
   18. transaksi                 (3,366)
   19. lagi                      (3,231)
   20. ada                       (3,211)
   21. ke                        (3,124)
   22. banget                    (3,023)
   23. pakai     